# Lesson 5 — Vector Spaces, Bases, Rank and Projections

**Notebook candidate v0.2.2 — automatic runtime pilot**

In a fresh Colab runtime, choose Run all. The setup downloads the exact pinned
runtime wheel automatically and checks its SHA-256 before installation.
No manual upload or desktop installation is required. Local users can keep
the same wheel in wheels/ beside the notebook; otherwise it is downloaded.

This pilot requires the public srai-math-v1.1.1rc1 release asset to be published
first. It is not yet an approved Lesson 5 release. NumPy, pandas and Matplotlib
must be available in the chosen environment (as in the tested Colab runtime).


## Learning objectives

1. Define vector spaces and subspaces.
2. Distinguish span, independence, basis, and dimension.
3. Compute coordinates and changes of basis.
4. Analyze column, row, and null spaces.
5. Construct orthogonal projections.
6. Interpret least squares geometrically.
7. Apply subspace ideas to AI and Decision Intelligence.


In [ ]:
# SRAI runtime: local wheel when available, otherwise a pinned public download.
from pathlib import Path
import hashlib, importlib, io, subprocess, sys, tempfile, urllib.request, zipfile

WHEEL_NAME = 'srai_math-1.1.1rc1-py3-none-any.whl'
WHEEL_SHA256 = '6e7033465ad3d9bf4650227a11be0380512a44fd476a83d5828ad4ec4f07e923'
WHEEL_URL = ('https://github.com/mbayekebe/srai-book-01-mathematical-foundations/'
             'releases/download/srai-math-v1.1.1rc1/' + WHEEL_NAME)

def ensure_srai_runtime():
    local = next((p for p in (Path.cwd()/'wheels'/WHEEL_NAME, Path.cwd()/WHEEL_NAME)
                  if p.is_file()), None)
    if local:
        data = local.read_bytes()
    else:
        try:
            request = urllib.request.Request(WHEEL_URL, headers={'User-Agent': 'SRAI-Notebook/1.0'})
            with urllib.request.urlopen(request, timeout=45) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError('Insecure download redirect rejected.')
                data = response.read(1024 * 1024 + 1)
        except Exception as exc:
            raise RuntimeError('SRAI runtime download unavailable. Check connectivity and that the '
                               'pinned release asset is published. No alternative version was installed. '
                               + WHEEL_URL) from exc
    if len(data) > 1024 * 1024 or hashlib.sha256(data).hexdigest() != WHEEL_SHA256:
        raise RuntimeError('SRAI wheel checksum mismatch. Stop; do not install this file.')
    previous = globals().get('_SRAI_PUBLIC_TARGET')
    loaded = [m for n,m in list(sys.modules.items()) if n=='srai_math' or n.startswith('srai_math.')]
    for module in loaded:
        origin = getattr(module, '__file__', None)
        if not previous or not origin or not Path(origin).resolve().is_relative_to(previous):
            raise RuntimeError('Another srai_math is loaded. Restart the kernel, then Run all.')
    if previous:
        target = previous
    else:
        work = Path(tempfile.mkdtemp(prefix='srai_runtime_'))
        wheel = work/WHEEL_NAME
        wheel.write_bytes(data)
        target = work/'site'
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index',
                               '--no-deps', '--no-compile', '--target', str(target), str(wheel)])
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        for entry in archive.infolist():
            if not entry.is_dir() and entry.filename.startswith('srai_math/'):
                path = target/entry.filename
                if not path.is_file() or path.read_bytes()!=archive.read(entry):
                    raise RuntimeError('Installed runtime differs from verified wheel. Restart kernel.')
    if str(target) not in sys.path:
        sys.path.insert(0, str(target))
    importlib.invalidate_caches()
    import srai_math
    if not Path(srai_math.__file__).resolve().is_relative_to(target):
        raise RuntimeError('Unexpected package import path. Restart kernel.')
    print('SRAI runtime READY: verified 1.1.1rc1 (' + ('local wheel' if local else 'public download') + ')')
    return target

_SRAI_PUBLIC_TARGET = ensure_srai_runtime()


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.algebra import (
    change_of_basis_matrix, column_space_basis, coordinates, is_in_span,
    null_space_basis, project_onto_subspace, projection_matrix, rank,
    reconstruct, row_space_basis,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()


## Span and subspaces

$$
\operatorname{span}\{\mathbf v_1,\ldots,\mathbf v_k\}
=
\left\{\sum_i c_i\mathbf v_i\right\}.
$$


In [ ]:
plane_basis=np.array([[1.,0.],[0.,1.],[0.,0.]])
inside=np.array([2.,3.,0.]); outside=np.array([2.,3.,1.])
assert is_in_span(inside,plane_basis)
assert not is_in_span(outside,plane_basis)
is_in_span(inside,plane_basis),is_in_span(outside,plane_basis)


## Independence, basis, and dimension

In [ ]:
independent=np.array([[1.,0.],[0.,1.]])
dependent=np.array([[1.,2.],[2.,4.]])
assert rank(independent)==2
assert rank(dependent)==1
rank(independent),rank(dependent)


## Coordinates in a basis

$$
B[\mathbf v]_B=\mathbf v.
$$


In [ ]:
B=np.array([[1.,1.],[1.,-1.]])
v=np.array([4.,2.])
c=coordinates(v,B)
assert np.allclose(reconstruct(c,B),v)
c


## Change of basis

In [ ]:
T=change_of_basis_matrix(np.eye(2),B)
c_new=T@v
assert np.allclose(B@c_new,v)
T,c_new


## Fundamental subspaces and rank-nullity

$$
\operatorname{rank}(A)+\operatorname{nullity}(A)=n.
$$


In [ ]:
A=np.array([[1.,2.,3.],[2.,4.,6.]])
C=column_space_basis(A); R=row_space_basis(A); N=null_space_basis(A)
r=rank(A); nullity=N.shape[1]
assert np.allclose(A@N,0,atol=1e-10)
assert r+nullity==A.shape[1]
{"rank":r,"nullity":nullity,"column_space":C.shape,"row_space":R.shape,"null_space":N.shape}


## Orthogonal projection

$$
P=QQ^\top,\qquad P^\top=P,\qquad P^2=P.
$$

Here $Q$ has orthonormal columns spanning the target subspace. Symmetry is
required for an **orthogonal** projector; idempotence alone also permits oblique projections.


In [ ]:
P=projection_matrix(plane_basis)
assert np.allclose(P,P.T)
assert np.allclose(P@P,P)
P


In [ ]:
x=np.array([2.,3.,4.])
x_proj=project_onto_subspace(x,plane_basis)
x_residual=x-x_proj
assert np.allclose(x_proj,[2.,3.,0.])
assert np.allclose(plane_basis.T@x_residual,0.)
x_proj,x_residual


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.axhline(0, color="0.75", linewidth=0.8)
ax.axvline(0, color="0.75", linewidth=0.8)
for start, delta, color, label in [
    ([0, 0], x[[0, 2]], "#193c63", "original vector"),
    ([0, 0], x_proj[[0, 2]], "#167d8d", "projection"),
    (x_proj[[0, 2]], x_residual[[0, 2]], "#b96824", "residual"),
]:
    ax.quiver(*start, *delta, angles="xy", scale_units="xy", scale=1,
              color=color, label=label)
ax.set(xlim=(-1, 5), ylim=(-1, 5), xlabel="First coordinate (x1)",
       ylabel="Third coordinate (x3)", title="Projection: first–third coordinate view")
ax.set_aspect("equal")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()
# The second coordinate is omitted: both x and its projection have x2 = 3.


The plot omits the second coordinate. In three dimensions the projection is $(2,3,0)^\top$, not $(2,0,0)^\top$.


## Least squares as projection

In [ ]:
A_ls=np.array([[1.,0.],[1.,1.],[1.,2.]])
b_ls=np.array([1.,2.,2.])
beta,*_=np.linalg.lstsq(A_ls,b_ls,rcond=None)
fitted=A_ls@beta
resid=b_ls-fitted
assert np.allclose(A_ls.T@resid,0,atol=1e-12)
beta,fitted,resid


## Statistics and AI interpretation

Least-squares fitted values are orthogonal projections onto a design matrix's
column space. PCA projects centered data onto an orthonormal principal subspace.
By contrast, a learned attention “projection” is generally just a linear map:
it need not be symmetric, idempotent, square, or an orthogonal projector.


## Decision Intelligence case — Policy subspace

In [ ]:
policy_basis=np.array([[1.,0.2],[0.2,1.],[0.6,0.6]])
proposal=np.array([0.9,0.7,0.1])
aligned=project_onto_subspace(proposal,policy_basis)
outside=proposal-aligned
assert np.allclose(policy_basis.T @ outside, 0, atol=1e-12)
assert np.allclose(aligned, [2/3, 7/15, 17/30])
pd.DataFrame({"proposed":proposal,"aligned":aligned,"outside_subspace":outside},
             index=["Agriculture","Health","Energy"])


The residual represents priorities not captured by the selected policy archetypes. It is a geometric diagnostic, not a normative judgment.


## Engineering notes

- Numerical rank depends on tolerance.
- Nearly dependent bases may be unstable.
- QR and SVD are preferred for robust projection work.
- Large systems should apply factorizations rather than explicitly forming the projection matrix.


## Exercises

### Level A
Explain span, basis, and rank.

### Level B
Verify projection identities and rank-nullity.

### Level C
Compare QR- and SVD-based projections.

### Capstone
Construct a policy subspace from historical policy profiles and interpret a proposed policy's residual.


## Key insight

A basis defines coordinates, rank measures independent information, and projection separates the explainable component from what lies outside a chosen model.
